### 1. Load Environment Variables

This cell loads the OpenAI API key from the local `.env` file.  
The key is stored in the variable `openai_api_key`, which will be used later for embeddings and the language model.

In [29]:
from dotenv import load_dotenv
import os

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("Set OPENAI_API_KEY in your .env file.")

### 2. Load the Excel Dataset as a LangChain Document

This cell reads the `Potential_Talents_Proxy_Labels.xlsx` file using pandas.  
The full dataframe is converted into text and wrapped inside a LangChain `Document` object so it can be used in the RAG pipeline.

In [30]:
from pathlib import Path
from langchain_core.documents import Document
import pandas as pd

document_path = Path("Potential_Talents_Proxy_Labels.xlsx")
df = pd.read_excel(document_path)
documents = [
    Document(
        page_content=df.to_string(),
        metadata={"source": str(document_path)},
    )
]

documents[0]

Document(metadata={'source': 'Potential_Talents_Proxy_Labels.xlsx'}, page_content='        id                                                                                                                                                                                                                  title              location  screening_score   match_label\n0        1                                                                                                innovative and driven professional seeking a role in data analyticsdata science in the information technology industry.         United States              100    high_match\n1        2                 ms applied data science student usc research assistant usc former data analytics intern at dr reddys laboratories former data science intern quadratyx actively seeking full time roles in summer 2025         United States              100    high_match\n2        3                                                                   

### 3. Split the Document into Smaller Chunks

This cell splits the full dataset text into smaller overlapping chunks.  
Chunking helps the retriever search only the most relevant parts of the dataset instead of sending the entire document to the model.

In [31]:
from langchain_core.documents import Document

def split_documents(documents, chunk_size=500, chunk_overlap=50):
    chunks = []
    step = chunk_size - chunk_overlap

    for document in documents:
        text = document.page_content
        for chunk_number, start in enumerate(range(0, len(text), step)):
            chunk_text = text[start : start + chunk_size]
            if not chunk_text.strip():
                continue

            chunks.append(
                Document(
                    page_content=chunk_text,
                    metadata={
                        **document.metadata,
                        "chunk": chunk_number,
                        "start_index": start,
                    },
                )
            )

    return chunks

document_chunks = split_documents(documents, chunk_size=500, chunk_overlap=50)
len(document_chunks), document_chunks[0]

(798,
 Document(metadata={'source': 'Potential_Talents_Proxy_Labels.xlsx', 'chunk': 0, 'start_index': 0}, page_content='        id                                                                                                                                                                                                                  title              location  screening_score   match_label\n0        1                                                                                                innovative and driven professional seeking a role in data analyticsdata science in the information technology indus'))

### 4. Create OpenAI Embeddings

This cell initializes the OpenAI embedding model.  
Embeddings convert text chunks into numerical vectors so that similar pieces of text can be searched using vector similarity.

In [32]:
from langchain_openai.embeddings import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

### 5. Store Document Chunks in a FAISS Vector Store

This cell creates a FAISS vector database from the document chunks and their embeddings.  
FAISS allows fast similarity search over the candidate dataset.

In [33]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(document_chunks, embeddings)

### 6. Create a Retriever

This cell converts the FAISS vector store into a retriever.  
The retriever will return the top 5 most similar chunks for each user query.

In [34]:
retriever = vector_store.as_retriever(
	 search_type="similarity",
	 search_kwargs={"k": 5}
	)

### 7. Build the RAG Question-Answering Chain

This cell creates the main RAG chain.  
The retriever finds relevant candidate information, the prompt formats the context and question, and the OpenAI model generates the final answer.

In [35]:
from langchain_openai import OpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

llm = OpenAI(api_key=openai_api_key)

prompt = PromptTemplate.from_template("""
'You are a recruiting screening model. '
'Read the candidate information and return only one label: '
'low_match, medium_match, or high_match.'.

Context:
{context}

Question:
{question}

Answer:
""")

qa_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

### 8. Ask a Question About the Candidate Dataset

This cell sends a query to the RAG chain and prints the model response.  
The query asks for the first candidate’s label, explanation, location, and title.

In [ ]:
from operator import itemgetter

qa_chain = (
    {
        "context": itemgetter("query") | retriever | format_docs,
        "question": itemgetter("query"),
    }
    | prompt
    | llm
    | StrOutputParser()
)

query = "return the label of the first candidate return the label and briefly explain your answer. and provide his location and title."
response = qa_chain.invoke({"query": query})
print(response)

The label for the first candidate in the dataset is high_match. The candidate is located in the United States and has the title of 'Innovative and Driven Professional Seeking a Role in Data Analytics/Data Science in the Information Technology Industry'. This candidate has a screening score of 95, which indicates a high potential for success in a data-related role. Their skills include expertise in Python, R, SQL, and Tableau, as well as a Master's degree in Data Science and a background in historical research. Overall, this candidate seems highly qualified and well-suited for a data-driven role in the United States.
